# Camera Pose Tester

Moves the arm to camera poses, prints the FK prediction (where the camera/gripper is and how far it points down), and shows a picture from the camera.

Poses are `[q1, q2, q3, q4, q5]` in degrees, same as `test_fk`. Edit them in `poses.json`.

FK comes from `Part1/DofbotKinematics.ipynb`, so keep this folder next to `Part1`.

In [ ]:
%run ../Part1/DofbotKinematics.ipynb

import json
import time
import cv2
import matplotlib.pyplot as plt
from Arm_Lib import Arm_Device

arm = Arm_Device()
cap = cv2.VideoCapture(0)   # try 1 if the picture is black

## Poses
Loaded from `poses.json`. Pitch from vertical: 180 = camera straight down, 90 = looking straight ahead.

In [ ]:
with open('poses.json') as f:
    poses = json.load(f)

for name, q in poses.items():
    Rot, Pot = fk_Dofbot(q)
    print(f"{name:12s} {q}  x y z (cm): {np.round(Pot*100, 1)}  pitch: {round(pitch_from_vertical(Rot), 1)}")

## Helper functions

In [ ]:
def move(q):
    # 6th value is the gripper (held at 90), 1500 = take 1.5 seconds
    arm.Arm_serial_servo_write6(*q, 90, 1500)
    time.sleep(2)


def take_picture():
    # Throw away a few old frames so we get the current view
    for _ in range(5):
        cap.read()
    ok, frame = cap.read()
    return frame if ok else None


def view(q, title=''):
    move(q)
    Rot, Pot = fk_Dofbot(q)
    print("\nAngles:", q)
    print("Predicted x y z (m):", np.round(Pot, 4))
    print("Predicted pitch from vertical (deg):", round(pitch_from_vertical(Rot), 1))

    frame = take_picture()
    if frame is None:
        print("No camera frame")
        return None
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title(f"{title} {q}")
    plt.axis('off')
    plt.show()
    return frame

## Try one saved pose
Change the name to any pose in `poses.json`.

In [ ]:
frame = view(poses['dealer'], 'dealer')

## Try all saved poses
Goes through every pose and saves each picture to `snaps/`.

In [ ]:
import os
os.makedirs('snaps', exist_ok=True)

for name, q in poses.items():
    frame = view(q, name)
    if frame is not None:
        cv2.imwrite(f"snaps/{name}.jpg", frame)

## Try your own angles
Edit `q` and rerun this cell as many times as you want.

In [ ]:
q = [90, 100, 30, 10, 90]
frame = view(q, 'custom')

## Save a pose you like
Saves `q` from the cell above into `poses.json`.

In [ ]:
name = 'new_pose'
poses[name] = q
with open('poses.json', 'w') as f:
    json.dump(poses, f, indent=2)
print("Saved", name, q)

## Done
Send the arm home and release the camera.

In [ ]:
move([90, 90, 90, 90, 90])
cap.release()